# 🚗 Vehicle Fuel Efficiency Prediction
## Notebook 8: Business Insights & Recommendations

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
df = pd.read_csv('../data/auto-mpg-cleaned.csv')
df_feat = pd.read_csv('../data/auto-mpg-features.csv')
final_model = joblib.load('../models/final_model.pkl')
print('Setup complete ✓')

---
## Insight 1 — Weight is the #1 Driver of Fuel Inefficiency

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
scatter = ax.scatter(df['weight'], df['mpg'],
                     c=df['cylinders'], cmap='RdYlGn_r',
                     s=60, alpha=0.7, edgecolors='white', linewidths=0.3)
z = np.polyfit(df['weight'], df['mpg'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['weight'].min(), df['weight'].max(), 200)
ax.plot(x_line, p(x_line), 'r--', linewidth=2.5, label='Trend')
plt.colorbar(scatter, ax=ax, label='# Cylinders')
ax.set_title('Insight 1: Vehicle Weight vs MPG (colored by Cylinders)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Vehicle Weight (lbs)')
ax.set_ylabel('MPG')
ax.legend()
plt.tight_layout()
plt.savefig('../plots/15_insight1_weight_mpg.png', dpi=150, bbox_inches='tight')
plt.show()
corr = df['weight'].corr(df['mpg'])
print(f'Pearson Correlation (Weight ↔ MPG): {corr:.4f}')
print('\n📌 Insight: Every 500 lbs reduction in weight corresponds to ~3–4 MPG improvement.')

---
## Insight 2 — Japanese & European Cars Are More Fuel Efficient

In [ ]:
origin_labels = {1: 'USA', 2: 'Europe', 3: 'Japan'}
df_o = df.copy()
df_o['origin_label'] = df_o['origin'].astype(int).map(origin_labels)
origin_stats = df_o.groupby('origin_label')['mpg'].agg(['mean', 'median', 'std']).round(2)
print('MPG by Origin:')
display(origin_stats)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
origin_stats['mean'].plot(kind='bar', ax=axes[0], color=['#e74c3c', '#3498db', '#2ecc71'],
                          edgecolor='white', rot=0)
axes[0].set_title('Avg MPG by Origin', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Avg MPG')
axes[0].set_xlabel('Origin')
groups = [df_o[df_o['origin_label'] == o]['mpg'].values for o in ['USA', 'Europe', 'Japan']]
bp = axes[1].boxplot(groups, patch_artist=True, labels=['USA', 'Europe', 'Japan'],
                     medianprops=dict(color='yellow', linewidth=2.5))
colors_bp = ['#e74c3c', '#3498db', '#2ecc71']
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('MPG Distribution by Origin', fontsize=13, fontweight='bold')
axes[1].set_ylabel('MPG')
plt.tight_layout()
plt.savefig('../plots/16_insight2_origin_mpg.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n📌 Insight: Japanese cars average ~30 MPG vs ~20 MPG for US cars — a 50% efficiency advantage.')

---
## Insight 3 — Fuel Efficiency Improved Dramatically Post-1973 Oil Crisis

In [ ]:
year_stats = df.groupby('model_year')['mpg'].agg(['mean', 'std']).reset_index()
year_stats['year_actual'] = year_stats['model_year'] + 1900
fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(year_stats['year_actual'],
                year_stats['mean'] - year_stats['std'],
                year_stats['mean'] + year_stats['std'],
                alpha=0.2, color='steelblue', label='±1 Std Dev')
ax.plot(year_stats['year_actual'], year_stats['mean'], 'o-',
        color='steelblue', linewidth=2.5, markersize=8, label='Avg MPG')
ax.axvline(1973, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.text(1973.2, ax.get_ylim()[1] * 0.95, '1973 Oil Crisis', color='red', fontsize=10)
ax.set_title('Insight 3: Average MPG Trend by Model Year', fontsize=14, fontweight='bold')
ax.set_xlabel('Model Year')
ax.set_ylabel('Average MPG')
ax.legend()
plt.tight_layout()
plt.savefig('../plots/17_insight3_year_trend.png', dpi=150, bbox_inches='tight')
plt.show()
pre_crisis  = df[df['model_year'] <= 73]['mpg'].mean()
post_crisis = df[df['model_year'] >= 74]['mpg'].mean()
print(f'Pre-crisis avg MPG  (70–73): {pre_crisis:.2f}')
print(f'Post-crisis avg MPG (74–82): {post_crisis:.2f}')
print(f'\n📌 Insight: MPG improved by {((post_crisis - pre_crisis)/pre_crisis*100):.1f}% after the 1973 oil shock — regulations and smaller engines drove efficiency gains.')

---
## Insight 4 — Power-to-Weight Ratio Captures True Efficiency Profile

In [ ]:
df_feat_plot = df_feat.copy()
df_feat_plot['origin_label'] = df['origin'].astype(int).map({1:'USA', 2:'Europe', 3:'Japan'})
fig, ax = plt.subplots(figsize=(11, 7))
origin_colors = {'USA': '#e74c3c', 'Europe': '#3498db', 'Japan': '#2ecc71'}
for origin, group in df_feat_plot.groupby('origin_label'):
    ax.scatter(group['power_to_weight'], group['mpg'],
               label=origin, color=origin_colors[origin],
               alpha=0.65, s=50, edgecolors='white', linewidths=0.3)
ax.set_title('Insight 4: Power-to-Weight Ratio vs MPG by Origin',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Horsepower / Weight (Power-to-Weight Ratio)')
ax.set_ylabel('MPG')
ax.legend(title='Origin', fontsize=10)
plt.tight_layout()
plt.savefig('../plots/18_insight4_power_weight.png', dpi=150, bbox_inches='tight')
plt.show()
print('📌 Insight: US cars cluster at high power-to-weight with low MPG; Japanese cars achieve lower power-to-weight but superior efficiency.')

---
## Insight 5 — Model Can Score Real Fleets (SHAP-style Summary)

In [ ]:
from sklearn.model_selection import train_test_split
TARGET = 'mpg'
FEATURE_COLS = [c for c in df_feat.columns if c != TARGET]
X = df_feat[FEATURE_COLS]
y = df_feat[TARGET]
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
y_pred = final_model.predict(X_test)
fleet_df = X_test.copy()
fleet_df['Actual MPG'] = y_test.values
fleet_df['Predicted MPG'] = y_pred.round(2)
fleet_df['Error'] = (fleet_df['Predicted MPG'] - fleet_df['Actual MPG']).round(2)
fleet_df['Efficiency Band'] = pd.cut(fleet_df['Predicted MPG'],
                                      bins=[0, 20, 28, 45],
                                      labels=['Low (<20)', 'Mid (20–28)', 'High (>28)'])
print('Sample Fleet Scoring:')
display(fleet_df[['Actual MPG', 'Predicted MPG', 'Error', 'Efficiency Band']].head(15))
band_counts = fleet_df['Efficiency Band'].value_counts()
fig, ax = plt.subplots(figsize=(8, 5))
band_counts.plot(kind='bar', ax=ax, color=['#e74c3c', '#f39c12', '#2ecc71'], edgecolor='white', rot=0)
ax.set_title('Fleet Efficiency Band Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../plots/19_fleet_scoring.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Recommendations for Motorq / Fleet Operators

| # | Recommendation | Business Value |
|---|---------------|----------------|
| 1 | **Retire high-weight vehicles first** — weight is #1 MPG driver | Direct fuel cost savings |
| 2 | **Prefer 4-cylinder vehicles for urban fleets** | ~40–50% better MPG than 8-cylinder |
| 3 | **Source from Japan/European OEMs** for high-efficiency segments | Strategic procurement |
| 4 | **Use the ML model to predict MPG before purchase** | ROI forecasting pre-acquisition |
| 5 | **Monitor power-to-weight ratio** as a fleet-wide efficiency KPI | Continuous monitoring |
| 6 | **Flag vehicles predicted >3 MPG below fleet average** for inspection | Driver coaching & maintenance alerts |
| 7 | **Use model year as a proxy for tech generation** in upgrade cycles | Long-term fleet planning |

---

## 10. Motorq-Specific Applications

> **Motorq connects to vehicle data at scale (OBD-II, telematics).** This MPG prediction model can be deployed as:

1. **Real-time efficiency scorer** — score every vehicle in a fleet daily
2. **Anomaly detector** — flag when actual MPG drops > 15% below predicted (engine issue, harsh driving)
3. **Procurement intelligence** — rank candidate vehicles by predicted MPG before purchase
4. **Carbon footprint estimator** — MPG → CO₂g/mile → ESG reporting
5. **Driver coaching** — predict expected vs actual MPG per driver to surface coaching opportunities